# Books RAG

This mini-project is part of homework from week 1 and implements a RAG to ask questions about books from Allen Downey (Green Tea Press). The project extracts text from the book PDF files, chunks them, and uses them to build a RAG system.

## Preliminaries

In [1]:
!uv add 'markitdown[pdf]'

Resolved 140 packages in 39ms                                        
⠙ Preparing packages... (0/17)                                                  ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/17)------------------     0 B/26.04 KiB           
⠙ Preparing packages... (0/17)m------------- 14.80 KiB/26.04 KiB         
⠙ Preparing packages... (0/17)m------------- 14.80 KiB/26.04 KiB         
flatbuffers          ------------------------------ 14.80 KiB/26.04 KiB
⠙ Preparing packages... (0/17)------------------     0 B/6.01 MiB            
flatbuffers          ------------------------------ 14.80 KiB/26.04 KiB
⠙ Preparing packages... (0/17)------------------     0 B/6.01 MiB            
flatbuffers          ------------------------------ 14.80 KiB/26.04 KiB
⠙ Preparing packages... (0/17)------------------     0 B/6.01 MiB            
flatbuffers          ------------------------------ 14.80 KiB/26.04 KiB
mpmath               

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

## 1. Download books

In [ ]:
import os

if not os.path.exists("books.csv"):
    !wget https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv

In [5]:
import download_books

download_books.download_books()

[skip] Think Python 2e already downloaded
[skip] Think DSP already downloaded
[skip] Think Complexity 2e already downloaded
[skip] Think Java 2e already downloaded
[skip] Physical Modeling in MATLAB already downloaded
[skip] Think OS already downloaded
[skip] Think C++ already downloaded


In [10]:
# HOMEWORK QUESTION 1
from markitdown import MarkItDown

md = MarkItDown()
result = md.convert("books/Think_Python_2e.pdf")
print("Lines", len(result.text_content.splitlines()))
print("Characters", len(result.text_content))
print(result.text_content[:100])

Lines 16511
Characters 450161
Think Python

How to Think Like a Computer Scientist

2nd Edition, Version 2.4.0

Think Python

Ho


In [14]:
!wc -l "books_text/Think_Python_2e.md"

   16268 books_text/Think_Python_2e.md


In [7]:
import convert_books_to_markdown

convert_books_to_markdown.convert_books()

[skip] Physical_Modeling_in_MATLAB.pdf already converted
[skip] Think_C__.pdf already converted
[skip] Think_Complexity_2e.pdf already converted
[skip] Think_DSP.pdf already converted
[skip] Think_Java_2e.pdf already converted
[skip] Think_OS.pdf already converted
[skip] Think_Python_2e.pdf already converted


## 2. Chunking the documents

In [15]:
!uv add gitsource

Resolved 140 packages in 11ms
Checked 134 packages in 76ms


In [101]:
from gitsource import chunk_documents

documents = [] 

for file in os.listdir('books_text'): 
    if file.endswith('.md'): 
        with open(f'books_text/{file}', 'r') as f: 
            lines = [line.strip() for line in f if line.strip()] 
            documents.append({ 'source': file, 'content': lines })


chunked_docs = chunk_documents(documents=documents, size=100, step=50) 

In [102]:
# HOMEWORK QUESTION 2

n_chunks = 0
for doc in chunked_docs:
    if doc["source"] == "Think_Python_2e.md":
        n_chunks += 1

print("Number of chunks in Think_Python_2e.md:", n_chunks)

# Example chunk
for doc in chunked_docs:
    if doc["source"] == "Think_Python_2e.md":
        print("Source:", doc['source'])
        print("Number of items:", len(doc['content']))
        print("First 5 items in chunk:\n", doc['content'][:5])
        print("Characters in chunk:", sum(len(item) for item in doc['content']))
        print("Words in chunk:", sum(len(item.split()) for item in doc['content']))
        break

Number of chunks in Think_Python_2e.md: 214
Source: Think_Python_2e.md
Number of items: 100
First 5 items in chunk:
 ['Think Python', 'How to Think Like a Computer Scientist', '2nd Edition, Version 2.4.0', 'Think Python', 'How to Think Like a Computer Scientist']
Characters in chunk: 5870
Words in chunk: 1020


## 3. Indexing with minsearch

In [103]:
!uv add minsearch

Resolved 140 packages in 10ms
Checked 134 packages in 252ms


In [104]:
from minsearch import Index

document_chunks = []

for chunk in chunked_docs: 
    chunk['content'] = '\n'.join(chunk['content']) 

index = Index(text_fields=['content'], keyword_fields=['source']) 
index.fit(chunked_docs) 

# HOMEWORK QUESTION 3
print("Total number of indexed chunks:", len(chunked_docs))

Total number of indexed chunks: 1009


## 4. Searching and RAG

In [ ]:
# HOMEWORK QUESTION 4
index.search("python function definition", num_results=5)

[{'start': 1900,
  'content': 'when you are comfortable with Python, I’ll make suggestions for installing Python on your\ncomputer.\nThere are a number of web pages you can use to run Python. If you already have a fa-\nvorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed\ninstructions for getting started at http://tinyurl.com/thinkpython2e.\nThere are two versions of Python, called Python 2 and Python 3. They are very similar, so\nif you learn one, it is easy to switch to the other. In fact, there are only a few differences you\nwill encounter as a beginner. This book is written for Python 3, but I include some notes\nabout Python 2.\nThe Python interpreter is a program that reads and executes Python code. Depending\non your environment, you might start the interpreter by clicking on an icon, or by typing\npython on a command line. When it starts, you should see output like this:\nPython 3.4.0 (default, Jun 19 2015, 14:20:21)\n[GCC 4.8.2] on linux\nType

## 5. Full RAG

In [111]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    response_dict = {
        "output_text": response.output_text,
        "usage": response.usage,
    }

    return response_dict

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [112]:
# HOMEWORK QUESTION 5
rag("python function definition")

{'output_text': "To define a function in Python, you use the `def` keyword followed by the function name and parentheses containing any parameters. The basic structure is as follows:\n\n```python\ndef function_name(parameters):\n    # body of the function\n    # perform operations\n    return result  # optional return statement\n```\n\n### Example:\nHere's an example function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    return a + b\n```\n\nYou can call this function with two arguments like this:\n\n```python\nresult = add_numbers(5, 3)\nprint(result)  # Output: 8\n```\n\nIn the context you provided, it mentions that functions allow you to define inputs and outputs carefully to avoid unexpected interactions, which helps in managing variables more effectively. When you define a function, the variables defined inside it (local variables) only exist while the function is running, avoiding variable name collisions in your scripts.",
 'usage': ResponseUsage(input_tokens=

## 6. Structured output

In [113]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [114]:
def llm_structured(user_prompt, output_type, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    response_dict = {
        "output_text": response.output_text,
        "usage": response.usage,
    }

    return response_dict

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm_structured(prompt, RAGResponse, instructions)
    return answer

In [115]:
# HOMEWORK QUESTION 6
rag("python function definition")

{'output_text': '{"answer":"In Python, a function is defined using the `def` keyword, followed by the function name and parentheses that may include parameters. The first line of the function definition is called the function header and should end with a colon. The body of the function is indented below the header, containing the statements that define what the function does.\\n\\n### Example of Function Definition in Python:\\n```python\\ndef my_function(parameter1, parameter2):\\n    \\"\\"\\"This is a docstring that describes the function.\\"\\"\\"\\n    result = parameter1 + parameter2  # some operations\\n    return result\\n```\\n\\nIn this example:\\n- `my_function` is the name of the function.\\n- `parameter1` and `parameter2` are the inputs (arguments) of the function.\\n- The `return` statement outputs the result of the function after executing its code. \\n\\n### Calling the Function:\\nTo invoke or call the function, use its name along with arguments in parentheses:\\n```py